In [ ]:
!pip -q install GEOparse

import os
import re
import gzip
import shutil
import requests
import warnings
from io import StringIO

import GEOparse
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [ ]:
def geo_series_prefix(acc):
    num = acc.replace("GSE", "")
    return f"GSE{num[:-3]}nnn"


def download_series_matrix(acc):
    filename = f"{acc}_series_matrix.txt.gz"
    prefix = geo_series_prefix(acc)

    ftp_url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{prefix}/{acc}/matrix/{filename}"
    alt_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?acc={acc}&format=file&file={filename}"

    if not os.path.exists(filename):
        r = requests.get(ftp_url, stream=True, timeout=60)
        if r.status_code == 200:
            with open(filename, "wb") as f:
                shutil.copyfileobj(r.raw, f)
        else:
            r = requests.get(alt_url, stream=True, timeout=60)
            r.raise_for_status()
            with open(filename, "wb") as f:
                shutil.copyfileobj(r.raw, f)

    return filename


def read_text_maybe_gzip(filepath):
    try:
        with gzip.open(filepath, "rt", encoding="utf-8", errors="ignore") as f:
            txt = f.read()
        if txt.strip():
            return txt
    except:
        pass

    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()


def parse_series_matrix(filepath):
    text = read_text_maybe_gzip(filepath)

    if "!series_matrix_table_begin" not in text:
        raise ValueError(f"{filepath} does not look like a valid series matrix file.")

    lines = text.splitlines()
    meta_dict = {}
    table_lines = []
    in_table = False

    for line in lines:
        if line.startswith("!series_matrix_table_begin"):
            in_table = True
            continue
        if line.startswith("!series_matrix_table_end"):
            in_table = False
            continue

        if in_table:
            table_lines.append(line)
        elif line.startswith("!Sample_"):
            parts = line.split("\t")
            key = parts[0].replace("!Sample_", "")
            vals = [p.strip().strip('"') for p in parts[1:]]
            meta_dict[key] = vals

    expr = pd.read_csv(StringIO("\n".join(table_lines)), sep="\t")
    expr = expr.rename(columns={expr.columns[0]: "Probe"})
    expr["Probe"] = expr["Probe"].astype(str)
    expr = expr.set_index("Probe")

    meta = pd.DataFrame(meta_dict)
    if "geo_accession" in meta.columns:
        meta.index = meta["geo_accession"]
    else:
        meta.index = expr.columns

    meta = meta.loc[expr.columns]
    return expr, meta


def load_gpl_table(gpl_id):
    gpl = GEOparse.get_GEO(geo=gpl_id, destdir=".")
    return gpl.table.copy()


def find_symbol_column(gpl_table):
    candidates = [
        "Gene Symbol", "Gene symbol", "Symbol", "SYMBOL",
        "ILMN_Gene", "GeneSymbol", "GENE_SYMBOL"
    ]
    for c in candidates:
        if c in gpl_table.columns:
            return c
    raise ValueError(f"No symbol column found. Columns: {gpl_table.columns.tolist()}")


def clean_symbol_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x == "" or x.lower() in {"nan", "---", "na"}:
        return np.nan
    x = re.split(r"///|//|;|,", x)[0].strip()
    if x == "":
        return np.nan
    return x


def map_probes_to_genes(expr_probe, gpl_table):
    symbol_col = find_symbol_column(gpl_table)

    annot = gpl_table[["ID", symbol_col]].copy()
    annot[symbol_col] = annot[symbol_col].apply(clean_symbol_value)
    annot = annot.dropna().drop_duplicates(subset=["ID"])

    temp = expr_probe.reset_index().rename(columns={"Probe": "ID"})
    merged = temp.merge(annot, on="ID", how="inner")

    sample_cols = [c for c in merged.columns if c not in ["ID", symbol_col]]
    for c in sample_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

    gene_expr = merged[[symbol_col] + sample_cols].groupby(symbol_col).mean()
    gene_expr.index.name = "Gene"
    return gene_expr


def infer_label_from_metadata(meta):
    combined_text = meta.astype(str).agg(" | ".join, axis=1).str.lower()

    label = pd.Series(index=meta.index, dtype="float")
    label[combined_text.str.contains("sarcoid")] = 1
    label[combined_text.str.contains("control|healthy|normal")] = 0

    meta2 = meta.copy()
    meta2["label"] = label
    meta2 = meta2.dropna(subset=["label"]).copy()
    meta2["label"] = meta2["label"].astype(int)
    return meta2


def load_microarray_cohort(acc, gpl_id, cohort_name):
    filepath = download_series_matrix(acc)
    expr_probe, meta = parse_series_matrix(filepath)
    meta2 = infer_label_from_metadata(meta)

    keep_samples = meta2.index.tolist()
    expr_probe = expr_probe.loc[:, keep_samples]

    gpl_table = load_gpl_table(gpl_id)
    expr_gene = map_probes_to_genes(expr_probe, gpl_table)

    y = meta2["label"].copy()
    y.index.name = "Sample"

    print(f"\n{cohort_name}")
    print("Probe-level shape:", expr_probe.shape)
    print("Gene-level shape :", expr_gene.shape)
    print("Labels:")
    print(y.value_counts().sort_index())

    return expr_gene, y, meta2

In [ ]:
expr18781, y18781, meta18781 = load_microarray_cohort(
    acc="GSE18781",
    gpl_id="GPL570",
    cohort_name="GSE18781"
)

expr42834, y42834, meta42834 = load_microarray_cohort(
    acc="GSE42834",
    gpl_id="GPL10558",
    cohort_name="GSE42834"
)

expr83456, y83456, meta83456 = load_microarray_cohort(
    acc="GSE83456",
    gpl_id="GPL10558",
    cohort_name="GSE83456"
)

31-Mar-2026 00:14:55 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
31-Mar-2026 00:14:55 INFO GEOparse - Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL570&form=text&view=full to ./GPL570.txt
INFO:GEOparse:Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL570&form=text&view=full to ./GPL570.txt
31-Mar-2026 00:14:57 DEBUG downloader - Total size: 0
DEBUG:GEOparse:Total size: 0
31-Mar-2026 00:14:57 DEBUG downloader - md5: None
DEBUG:GEOparse:md5: None
81.6MB [00:02, 34.1MB/s]
31-Mar-2026 00:15:00 DEBUG downloader - Moving /tmp/tmphylxt9w6 to /content/GPL570.txt
DEBUG:GEOparse:Moving /tmp/tmphylxt9w6 to /content/GPL570.txt
31-Mar-2026 00:15:00 DEBUG downloader - Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL570&form=text&view=full
DEBUG:GEOparse:Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL570&form


GSE18781
Probe-level shape: (54675, 55)
Gene-level shape : (22880, 55)
Labels:
label
0    55
Name: count, dtype: int64


31-Mar-2026 00:15:11 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
31-Mar-2026 00:15:11 INFO GEOparse - Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL10558&form=text&view=full to ./GPL10558.txt
INFO:GEOparse:Downloading http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL10558&form=text&view=full to ./GPL10558.txt
31-Mar-2026 00:15:12 DEBUG downloader - Total size: 0
DEBUG:GEOparse:Total size: 0
31-Mar-2026 00:15:12 DEBUG downloader - md5: None
DEBUG:GEOparse:md5: None
72.3MB [00:02, 25.4MB/s]
31-Mar-2026 00:15:15 DEBUG downloader - Moving /tmp/tmpr3kkehhr to /content/GPL10558.txt
DEBUG:GEOparse:Moving /tmp/tmpr3kkehhr to /content/GPL10558.txt
31-Mar-2026 00:15:15 DEBUG downloader - Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&acc=GPL10558&form=text&view=full
DEBUG:GEOparse:Successfully downloaded http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?targ=self&a


GSE42834
Probe-level shape: (47323, 294)
Gene-level shape : (31426, 294)
Labels:
label
0    189
1    105
Name: count, dtype: int64


31-Mar-2026 00:15:23 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
31-Mar-2026 00:15:23 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
31-Mar-2026 00:15:23 INFO GEOparse - Parsing ./GPL10558.txt: 
INFO:GEOparse:Parsing ./GPL10558.txt: 
31-Mar-2026 00:15:23 DEBUG GEOparse - PLATFORM: GPL10558
DEBUG:GEOparse:PLATFORM: GPL10558



GSE83456
Probe-level shape: (47231, 202)
Gene-level shape : (31334, 202)
Labels:
label
0    202
Name: count, dtype: int64


In [ ]:
print("\nGSE18781")
print(y18781.value_counts().sort_index())

print("\nGSE42834")
print(y42834.value_counts().sort_index())

print("\nGSE83456")
print(y83456.value_counts().sort_index())

y_total = pd.concat([y18781, y42834, y83456])

print("\nCombined")
print(y_total.value_counts().sort_index())
print("Total samples:", len(y_total))


GSE18781
label
0    55
Name: count, dtype: int64

GSE42834
label
0    189
1    105
Name: count, dtype: int64

GSE83456
label
0    202
Name: count, dtype: int64

Combined
label
0    446
1    105
Name: count, dtype: int64
Total samples: 551


In [ ]:
shared_genes = sorted(
    set(expr18781.index) &
    set(expr42834.index) &
    set(expr83456.index)
)

print("Shared genes:", len(shared_genes))

X_all = pd.concat(
    [
        expr18781.loc[shared_genes],
        expr42834.loc[shared_genes],
        expr83456.loc[shared_genes]
    ],
    axis=1
)

y_all = pd.concat([y18781, y42834, y83456])

# transpose so rows = samples, columns = genes
X_all = X_all.T
X_all = X_all.loc[y_all.index]

groups = np.array(
    ["GSE18781"] * len(y18781) +
    ["GSE42834"] * len(y42834) +
    ["GSE83456"] * len(y83456)
)

print("Final merged X shape:", X_all.shape)
print("Final merged y shape:", y_all.shape)
print(y_all.value_counts().sort_index())

Shared genes: 16622
Final merged X shape: (551, 16622)
Final merged y shape: (551,)
label
0    446
1    105
Name: count, dtype: int64


In [ ]:
summary_df = pd.DataFrame([
    ["GSE18781", int((y18781 == 0).sum()), int((y18781 == 1).sum()), len(y18781)],
    ["GSE42834", int((y42834 == 0).sum()), int((y42834 == 1).sum()), len(y42834)],
    ["GSE83456", int((y83456 == 0).sum()), int((y83456 == 1).sum()), len(y83456)],
    ["Combined", int((y_all == 0).sum()), int((y_all == 1).sum()), len(y_all)],
], columns=["Cohort", "Controls", "Sarcoidosis", "Total"])

summary_df

,Cohort,Controls,Sarcoidosis,Total
0,GSE18781,55,0,55
1,GSE42834,189,105,294
2,GSE83456,202,0,202
3,Combined,446,105,551


In [ ]:
for name, meta in [("GSE18781", meta18781), ("GSE42834", meta42834), ("GSE83456", meta83456)]:
    print("\n" + "="*60)
    print(name)
    print("="*60)
    for col in meta.columns:
        vals = meta[col].astype(str).dropna().unique()
        vals = [v for v in vals if v != "nan"]
        if len(vals) <= 20:
            print(f"\nCOLUMN: {col}")
            print(vals[:20])


GSE18781

COLUMN: status
['Public on Nov 04 2009']

COLUMN: submission_date
['Oct 28 2009']

COLUMN: last_update_date
['Aug 28 2018']

COLUMN: type
['RNA']

COLUMN: channel_count
['1']

COLUMN: source_name_ch1
['peripheral blood']

COLUMN: organism_ch1
['Homo sapiens']

COLUMN: characteristics_ch1
['disease state: Axial Spondyloarthropathy', 'disease state: Control', 'disease state: Sarcoidosis']

COLUMN: treatment_protocol_ch1
['Whole blood was collected and incubated in 4 PAXGene tubes/subject (PreAnalytiX, a Qiagen BD Company, Valencia, CA, USA) for two hours and stored frozen at -80C until the RNA was isolated.']

COLUMN: molecule_ch1
['total RNA']

COLUMN: extract_protocol_ch1
["Total RNA isolation was performed using the PAXgene Blood RNA Isolation System (PreAnalytiX, a Qiagen BD Company, Valencia, CA, USA) following the manufacturer's recommendations.  Following RNA isolation, DNase-treatment was performed as per PAXGene manufacturer’s recommendation.  RNA recovery and quality

In [ ]:
for name, meta in [("GSE18781", meta18781), ("GSE42834", meta42834), ("GSE83456", meta83456)]:
    print("\n" + "="*60)
    print(name)
    print("="*60)
    for col in meta.columns:
        vals = meta[col].astype(str).str.lower()
        hits = vals[vals.str.contains("sarco|control|healthy|normal|tb|tuberculosis|disease", na=False)].unique()
        if len(hits) > 0:
            print(f"\nCOLUMN: {col}")
            print(hits[:20])


GSE18781

COLUMN: title
['blood - control - 017a' 'blood - control - 020a'
 'blood - control - 021a' 'blood - control - 022a'
 'blood - control - 023a' 'blood - control - 032a'
 'blood - control - 051a' 'blood - control - 052a'
 'blood - control - 055a' 'blood - control - 056a'
 'blood - control - 065a' 'blood - control - 067a'
 'blood - control - 088a' 'blood - control - 090a'
 'blood - control - 092a' 'blood - control - 111a'
 'blood - control - 112a' 'blood - control - 138a'
 'blood - control - 140a' 'blood - control - 141a']

COLUMN: characteristics_ch1
['disease state: axial spondyloarthropathy' 'disease state: control'
 'disease state: sarcoidosis']

COLUMN: hyb_protocol
['labeled crna targets were chemically fragmented at 95c for 35 min as per affymetrix’s recommendations. the fragmented material was combined with affymetrix hybridization controls: biotinylated control oligomer (b2) and biotinylated control crnas for biob, bioc, biod and crex (affymetrix) in hybridization buffe

In [ ]:
meta18781_text = meta18781.astype(str).agg(" | ".join, axis=1).str.lower()

keep18781 = meta18781_text.str.contains("sarco", na=False) | meta18781_text.str.contains("control", na=False)
y18781 = pd.Series(index=meta18781.index, dtype="int")

y18781[meta18781_text.str.contains("sarco", na=False)] = 1
y18781[meta18781_text.str.contains("control", na=False)] = 0

y18781 = y18781[keep18781].dropna().astype(int)
expr18781 = expr18781.loc[:, y18781.index]

print(y18781.value_counts())

0    55
Name: count, dtype: int64


In [ ]:
meta42834_text = meta42834.astype(str).agg(" | ".join, axis=1).str.lower()

keep42834 = meta42834_text.str.contains("sarco", na=False) | meta42834_text.str.contains("control", na=False)
y42834 = pd.Series(index=meta42834.index, dtype="int")

y42834[meta42834_text.str.contains("sarco", na=False)] = 1
y42834[meta42834_text.str.contains("control", na=False)] = 0

y42834 = y42834[keep42834].dropna().astype(int)
expr42834 = expr42834.loc[:, y42834.index]

print(y42834.value_counts())

0    143
1    141
Name: count, dtype: int64


In [ ]:
meta83456_text = meta83456.astype(str).agg(" | ".join, axis=1).str.lower()

keep83456 = meta83456_text.str.contains("sarco", na=False) | meta83456_text.str.contains("control", na=False)
y83456 = pd.Series(index=meta83456.index, dtype="int")

y83456[meta83456_text.str.contains("sarco", na=False)] = 1
y83456[meta83456_text.str.contains("control", na=False)] = 0

y83456 = y83456[keep83456].dropna().astype(int)
expr83456 = expr83456.loc[:, y83456.index]

print(y83456.value_counts())

0    61
1    49
Name: count, dtype: int64


In [ ]:
summary_df = pd.DataFrame([
    ["GSE18781", int((y18781 == 0).sum()), int((y18781 == 1).sum()), len(y18781)],
    ["GSE42834", int((y42834 == 0).sum()), int((y42834 == 1).sum()), len(y42834)],
    ["GSE83456", int((y83456 == 0).sum()), int((y83456 == 1).sum()), len(y83456)],
], columns=["Cohort", "Controls", "Sarcoidosis", "Total"])

summary_df.loc[len(summary_df)] = [
    "Combined",
    summary_df["Controls"].sum(),
    summary_df["Sarcoidosis"].sum(),
    summary_df["Total"].sum()
]

summary_df

,Cohort,Controls,Sarcoidosis,Total
0,GSE18781,55,0,55
1,GSE42834,143,141,284
2,GSE83456,61,49,110
3,Combined,259,190,449


In [ ]:
print("GSE18781 labels:")
print(meta18781.head(20))

print("\nGSE42834 labels:")
print(meta42834.head(20))

print("\nGSE83456 labels:")
print(meta83456.head(20))

GSE18781 labels:
                                              title geo_accession  \
GSM465907  Blood - Axial Spondyloarthropathy - 015A     GSM465907   
GSM465908  Blood - Axial Spondyloarthropathy - 019A     GSM465908   
GSM465909  Blood - Axial Spondyloarthropathy - 029A     GSM465909   
GSM465910  Blood - Axial Spondyloarthropathy - 030A     GSM465910   
GSM465911  Blood - Axial Spondyloarthropathy - 033A     GSM465911   
GSM465912  Blood - Axial Spondyloarthropathy - 058A     GSM465912   
GSM465913  Blood - Axial Spondyloarthropathy - 062A     GSM465913   
GSM465914  Blood - Axial Spondyloarthropathy - 066A     GSM465914   
GSM465915  Blood - Axial Spondyloarthropathy - 069A     GSM465915   
GSM465916  Blood - Axial Spondyloarthropathy - 074A     GSM465916   
GSM465917  Blood - Axial Spondyloarthropathy - 083A     GSM465917   
GSM465918  Blood - Axial Spondyloarthropathy - 083B     GSM465918   
GSM465919  Blood - Axial Spondyloarthropathy - 089A     GSM465919   
GSM465920  Blood 

In [ ]:
for name, meta in [("GSE18781", meta18781), ("GSE42834", meta42834), ("GSE83456", meta83456)]:
    print("\n" + "="*50)
    print(name)
    print("="*50)

    for col in meta.columns:
        vals = meta[col].astype(str).str.lower()
        if vals.str.contains("sarco|control|healthy|normal|tb|disease", na=False).any():
            print("\nCOLUMN:", col)
            print(meta[col].unique()[:20])


GSE18781

COLUMN: title
['Blood - Axial Spondyloarthropathy - 015A'
 'Blood - Axial Spondyloarthropathy - 019A'
 'Blood - Axial Spondyloarthropathy - 029A'
 'Blood - Axial Spondyloarthropathy - 030A'
 'Blood - Axial Spondyloarthropathy - 033A'
 'Blood - Axial Spondyloarthropathy - 058A'
 'Blood - Axial Spondyloarthropathy - 062A'
 'Blood - Axial Spondyloarthropathy - 066A'
 'Blood - Axial Spondyloarthropathy - 069A'
 'Blood - Axial Spondyloarthropathy - 074A'
 'Blood - Axial Spondyloarthropathy - 083A'
 'Blood - Axial Spondyloarthropathy - 083B'
 'Blood - Axial Spondyloarthropathy - 089A'
 'Blood - Axial Spondyloarthropathy - 096A'
 'Blood - Axial Spondyloarthropathy - 097A'
 'Blood - Axial Spondyloarthropathy - 152A'
 'Blood - Axial Spondyloarthropathy - 159A'
 'Blood - Axial Spondyloarthropathy - 162A' 'Blood - Control - 017A'
 'Blood - Control - 020A']

COLUMN: characteristics_ch1
['disease state: Axial Spondyloarthropathy' 'disease state: Control'
 'disease state: Sarcoidosis']

C

In [ ]:
def clean_labels(meta, expr, cohort_name):
    text = meta.astype(str).agg(" | ".join, axis=1).str.lower()

    y = pd.Series(index=meta.index, dtype="float")

    # strict sarcoidosis
    y[text.str.contains("sarcoidosis")] = 1

    # strict control
    y[text.str.contains("healthy")] = 0

    # KEEP only those two groups
    keep = y.dropna().index

    y = y.loc[keep].astype(int)
    expr = expr.loc[:, keep]

    print(f"\n{cohort_name}")
    print(y.value_counts())

    return expr, y

In [ ]:
expr18781, y18781 = clean_labels(meta18781, expr18781, "GSE18781")
expr42834, y42834 = clean_labels(meta42834, expr42834, "GSE42834")
expr83456, y83456 = clean_labels(meta83456, expr83456, "GSE83456")


GSE18781
1    12
Name: count, dtype: int64

GSE42834
1    144
Name: count, dtype: int64

GSE83456
Series([], Name: count, dtype: int64)


In [ ]:
def show_label_candidates(meta, cohort_name):
    print(f"\n{'='*70}")
    print(cohort_name)
    print(f"{'='*70}")

    for col in meta.columns:
        vals = meta[col].astype(str).dropna().unique()
        hits = [v for v in vals if any(k in v.lower() for k in [
            "sarco", "control", "healthy", "normal", "tb", "tuberculosis", "disease"
        ])]
        if hits:
            print(f"\nCOLUMN: {col}")
            for v in hits[:30]:
                print(" ", v)

show_label_candidates(meta18781, "GSE18781")
show_label_candidates(meta42834, "GSE42834")
show_label_candidates(meta83456, "GSE83456")


GSE18781

COLUMN: title
  Blood - Control - 017A
  Blood - Control - 020A
  Blood - Control - 021A
  Blood - Control - 022A
  Blood - Control - 023A
  Blood - Control - 032A
  Blood - Control - 051A
  Blood - Control - 052A
  Blood - Control - 055A
  Blood - Control - 056A
  Blood - Control - 065A
  Blood - Control - 067A
  Blood - Control - 088A
  Blood - Control - 090A
  Blood - Control - 092A
  Blood - Control - 111A
  Blood - Control - 112A
  Blood - Control - 138A
  Blood - Control - 140A
  Blood - Control - 141A
  Blood - Control - 153A
  Blood - Control - 154A
  Blood - Control - 155A
  Blood - Control - 156A
  Blood - Control - 157A
  Blood - Sarcoidosis - 001A
  Blood - Sarcoidosis - 003A
  Blood - Sarcoidosis - 005A
  Blood - Sarcoidosis - 013A
  Blood - Sarcoidosis - 031A

COLUMN: characteristics_ch1
  disease state: Axial Spondyloarthropathy
  disease state: Control
  disease state: Sarcoidosis

COLUMN: hyb_protocol
  Labeled cRNA targets were chemically fragmented at 95C 

In [ ]:
meta.astype(str).agg(" | ".join, axis=1).str.lower()

,0
GSM2203540,blood human eptb rep1 | gsm2203540 | public on...
GSM2203541,blood human eptb rep2 | gsm2203541 | public on...
GSM2203542,blood human eptb rep3 | gsm2203542 | public on...
GSM2203543,blood human eptb rep4 | gsm2203543 | public on...
GSM2203544,blood human eptb rep5 | gsm2203544 | public on...
...,...
GSM2203737,blood human sarcoid rep45 | gsm2203737 | publi...
GSM2203738,blood human sarcoid rep46 | gsm2203738 | publi...
GSM2203739,blood human sarcoid rep47 | gsm2203739 | publi...
GSM2203740,blood human sarcoid rep48 | gsm2203740 | publi...


In [ ]:
def label_from_exact_rules(meta, expr, sarco_keywords, control_keywords, exclude_keywords=None, cohort_name="cohort"):
    # force metadata to strings and build one text blob per sample
    text = meta.fillna("").astype(str).apply(lambda row: " | ".join(row.values.astype(str)), axis=1).str.lower()

    if exclude_keywords:
        exclude_pattern = "|".join([re.escape(x.lower()) for x in exclude_keywords])
        text = text.mask(text.str.contains(exclude_pattern, na=False), "")

    sarco_pattern = "|".join([re.escape(x.lower()) for x in sarco_keywords])
    control_pattern = "|".join([re.escape(x.lower()) for x in control_keywords])

    y = pd.Series(index=text.index, dtype="float")
    y[text.str.contains(sarco_pattern, na=False)] = 1
    y[text.str.contains(control_pattern, na=False)] = 0

    keep = [s for s in y.dropna().index if s in expr.columns]

    y = y.loc[keep].astype(int)
    expr = expr.loc[:, keep]

    print(f"\n{cohort_name}")
    print(y.value_counts().sort_index())
    print("Total:", len(y))

    return expr, y

In [ ]:
meta18781_sub = meta18781.loc[meta18781.index.intersection(expr18781.columns)].copy()
meta42834_sub = meta42834.loc[meta42834.index.intersection(expr42834.columns)].copy()
meta83456_sub = meta83456.loc[meta83456.index.intersection(expr83456.columns)].copy()

In [ ]:
def show_label_candidates(meta, cohort_name):
    print(f"\n{'='*60}")
    print(cohort_name)
    print(f"{'='*60}")

    for col in meta.columns:
        vals = pd.Series(meta[col]).dropna().astype(str).unique()
        hits = [v for v in vals if any(k in v.lower() for k in [
            "sarco", "control", "healthy", "normal", "tb", "tuberculosis", "patient"
        ])]
        if hits:
            print(f"\nCOLUMN: {col}")
            for v in hits[:30]:
                print(v)

show_label_candidates(meta18781_sub, "GSE18781")
show_label_candidates(meta42834_sub, "GSE42834")
show_label_candidates(meta83456_sub, "GSE83456")


GSE18781

COLUMN: title
Blood - Sarcoidosis - 001A
Blood - Sarcoidosis - 003A
Blood - Sarcoidosis - 005A
Blood - Sarcoidosis - 013A
Blood - Sarcoidosis - 031A
Blood - Sarcoidosis - 034A
Blood - Sarcoidosis - 049A
Blood - Sarcoidosis - 050A
Blood - Sarcoidosis - 059A
Blood - Sarcoidosis - 064A
Blood - Sarcoidosis - 077A
Blood - Sarcoidosis - 086A

COLUMN: characteristics_ch1
disease state: Sarcoidosis

COLUMN: hyb_protocol
Labeled cRNA targets were chemically fragmented at 95C for 35 min as per Affymetrix’s recommendations. The fragmented material was combined with Affymetrix hybridization controls: biotinylated control oligomer (B2) and biotinylated control cRNAs for BioB, BioC, BioD and CreX (Affymetrix) in hybridization buffer. Following the manufacturer’s recommendations, 6.5 µg of target were hybridized with the Human Genome U133 Plus 2.0 GeneChip array (Affymetrix) for 16 hours at 45C. Post-hybridization array processing was performed on the Fluidics Station 450 (Affymetrix); arr

In [ ]:
def load_raw_cohort(acc):
    filepath = download_series_matrix(acc)
    expr_probe, meta = parse_series_matrix(filepath)
    return expr_probe, meta

In [ ]:
expr18781_raw, meta18781_raw = load_raw_cohort("GSE18781")
expr42834_raw, meta42834_raw = load_raw_cohort("GSE42834")
expr83456_raw, meta83456_raw = load_raw_cohort("GSE83456")

In [ ]:
def label_from_title(meta, expr, title_col="title", cohort_name="cohort"):
    title = meta[title_col].fillna("").astype(str).str.lower()

    y = pd.Series(index=meta.index, dtype="float")

    y[title.str.contains("sarcoid", na=False)] = 1
    y[title.str.contains("control", na=False)] = 0

    keep = [s for s in y.dropna().index if s in expr.columns]

    y = y.loc[keep].astype(int)
    expr = expr.loc[:, keep]

    print(f"\n{cohort_name}")
    print(y.value_counts().sort_index())
    print("Total:", len(y))

    return expr, y

In [ ]:
expr18781_f, y18781 = label_from_title(meta18781_raw, expr18781_raw, cohort_name="GSE18781")
expr42834_f, y42834 = label_from_title(meta42834_raw, expr42834_raw, cohort_name="GSE42834")
expr83456_f, y83456 = label_from_title(meta83456_raw, expr83456_raw, cohort_name="GSE83456")


GSE18781
0    25
1    12
Name: count, dtype: int64
Total: 37

GSE42834
0    143
1    108
Name: count, dtype: int64
Total: 251

GSE83456
0    61
1    49
Name: count, dtype: int64
Total: 110


In [ ]:
print(meta83456_raw["title"].head(50).tolist())

['Blood Human EPTB rep1', 'Blood Human EPTB rep2', 'Blood Human EPTB rep3', 'Blood Human EPTB rep4', 'Blood Human EPTB rep5', 'Blood Human EPTB rep6', 'Blood Human EPTB rep7', 'Blood Human EPTB rep8', 'Blood Human EPTB rep9', 'Blood Human EPTB rep10', 'Blood Human EPTB rep11', 'Blood Human EPTB rep12', 'Blood Human EPTB rep13', 'Blood Human EPTB rep14', 'Blood Human EPTB rep15', 'Blood Human EPTB rep16', 'Blood Human EPTB rep17', 'Blood Human EPTB rep18', 'Blood Human EPTB rep19', 'Blood Human EPTB rep20', 'Blood Human EPTB rep21', 'Blood Human EPTB rep22', 'Blood Human EPTB rep23', 'Blood Human EPTB rep24', 'Blood Human EPTB rep25', 'Blood Human EPTB rep26', 'Blood Human EPTB rep27', 'Blood Human EPTB rep28', 'Blood Human EPTB rep29', 'Blood Human EPTB rep30', 'Blood Human EPTB rep31', 'Blood Human EPTB rep32', 'Blood Human EPTB rep33', 'Blood Human EPTB rep34', 'Blood Human EPTB rep35', 'Blood Human EPTB rep36', 'Blood Human EPTB rep37', 'Blood Human EPTB rep38', 'Blood Human EPTB re

In [ ]:
for col in meta83456_raw.columns:
    vals = meta83456_raw[col].astype(str).dropna().unique()
    hits = [v for v in vals if any(k in v.lower() for k in ["sarco", "control", "healthy", "normal", "tb"])]
    if hits:
        print("\nCOLUMN:", col)
        print(hits[:30])


COLUMN: title
['Blood Human EPTB rep1', 'Blood Human EPTB rep2', 'Blood Human EPTB rep3', 'Blood Human EPTB rep4', 'Blood Human EPTB rep5', 'Blood Human EPTB rep6', 'Blood Human EPTB rep7', 'Blood Human EPTB rep8', 'Blood Human EPTB rep9', 'Blood Human EPTB rep10', 'Blood Human EPTB rep11', 'Blood Human EPTB rep12', 'Blood Human EPTB rep13', 'Blood Human EPTB rep14', 'Blood Human EPTB rep15', 'Blood Human EPTB rep16', 'Blood Human EPTB rep17', 'Blood Human EPTB rep18', 'Blood Human EPTB rep19', 'Blood Human EPTB rep20', 'Blood Human EPTB rep21', 'Blood Human EPTB rep22', 'Blood Human EPTB rep23', 'Blood Human EPTB rep24', 'Blood Human EPTB rep25', 'Blood Human EPTB rep26', 'Blood Human EPTB rep27', 'Blood Human EPTB rep28', 'Blood Human EPTB rep29', 'Blood Human EPTB rep30']

COLUMN: source_name_ch1
['Blood Human EPTB', 'Blood Human Control', 'Blood Human PTB', 'Blood Human Sarcoid']

COLUMN: data_processing
['quantile normalization, background subtraction using Genomestudio']


In [ ]:
print(meta83456_raw["title"].head(50).tolist())

['Blood Human EPTB rep1', 'Blood Human EPTB rep2', 'Blood Human EPTB rep3', 'Blood Human EPTB rep4', 'Blood Human EPTB rep5', 'Blood Human EPTB rep6', 'Blood Human EPTB rep7', 'Blood Human EPTB rep8', 'Blood Human EPTB rep9', 'Blood Human EPTB rep10', 'Blood Human EPTB rep11', 'Blood Human EPTB rep12', 'Blood Human EPTB rep13', 'Blood Human EPTB rep14', 'Blood Human EPTB rep15', 'Blood Human EPTB rep16', 'Blood Human EPTB rep17', 'Blood Human EPTB rep18', 'Blood Human EPTB rep19', 'Blood Human EPTB rep20', 'Blood Human EPTB rep21', 'Blood Human EPTB rep22', 'Blood Human EPTB rep23', 'Blood Human EPTB rep24', 'Blood Human EPTB rep25', 'Blood Human EPTB rep26', 'Blood Human EPTB rep27', 'Blood Human EPTB rep28', 'Blood Human EPTB rep29', 'Blood Human EPTB rep30', 'Blood Human EPTB rep31', 'Blood Human EPTB rep32', 'Blood Human EPTB rep33', 'Blood Human EPTB rep34', 'Blood Human EPTB rep35', 'Blood Human EPTB rep36', 'Blood Human EPTB rep37', 'Blood Human EPTB rep38', 'Blood Human EPTB re

In [ ]:
print("GSE18781")
print(y18781.value_counts())
print("Total:", len(y18781))

print("\nGSE42834")
print(y42834.value_counts())
print("Total:", len(y42834))

print("\nGSE83456")
print(y83456.value_counts())
print("Total:", len(y83456))

GSE18781
0    25
1    12
Name: count, dtype: int64
Total: 37

GSE42834
0    143
1    108
Name: count, dtype: int64
Total: 251

GSE83456
0    61
1    49
Name: count, dtype: int64
Total: 110


In [ ]:
y_all_temp = pd.concat([y18781, y42834, y83456])

print("\n=== FINAL TOTAL ===")
print(y_all_temp.value_counts())
print("Total samples:", len(y_all_temp))


=== FINAL TOTAL ===
0    229
1    169
Name: count, dtype: int64
Total samples: 398


In [ ]:
summary_df = pd.DataFrame([
    ["GSE18781", (y18781==0).sum(), (y18781==1).sum(), len(y18781)],
    ["GSE42834", (y42834==0).sum(), (y42834==1).sum(), len(y42834)],
    ["GSE83456", (y83456==0).sum(), (y83456==1).sum(), len(y83456)],
    ["Combined", (y_all_temp==0).sum(), (y_all_temp==1).sum(), len(y_all_temp)],
], columns=["Cohort", "Controls", "Sarcoidosis", "Total"])

summary_df

,Cohort,Controls,Sarcoidosis,Total
0,GSE18781,25,12,37
1,GSE42834,143,108,251
2,GSE83456,61,49,110
3,Combined,229,169,398


In [ ]:
print(y_all_temp.value_counts())
print(len(y_all_temp))

0    229
1    169
Name: count, dtype: int64
398


In [ ]:
def clean_gene_index(df):
    df = df.copy()
    df.index = (
        df.index.astype(str)
        .str.upper()
        .str.strip()
    )
    return df

expr18781_f = clean_gene_index(expr18781_f)
expr42834_f = clean_gene_index(expr42834_f)
expr83456_f = clean_gene_index(expr83456_f)

In [ ]:
print("Example genes 18781:", list(expr18781_f.index[:10]))
print("Example genes 42834:", list(expr42834_f.index[:10]))
print("Example genes 83456:", list(expr83456_f.index[:10]))

Example genes 18781: ['1007_S_AT', '1053_AT', '117_AT', '121_AT', '1255_G_AT', '1294_AT', '1316_AT', '1320_AT', '1405_I_AT', '1431_AT']
Example genes 42834: ['ILMN_1343291', 'ILMN_1343295', 'ILMN_1651199', 'ILMN_1651209', 'ILMN_1651210', 'ILMN_1651221', 'ILMN_1651228', 'ILMN_1651229', 'ILMN_1651230', 'ILMN_1651232']
Example genes 83456: ['ILMN_1343291', 'ILMN_1343295', 'ILMN_1651199', 'ILMN_1651209', 'ILMN_1651210', 'ILMN_1651221', 'ILMN_1651228', 'ILMN_1651229', 'ILMN_1651230', 'ILMN_1651232']


In [ ]:
import GEOparse

gpl570 = GEOparse.get_GEO("GPL570", destdir=".")
gpl570_table = gpl570.table

# find symbol column
print(gpl570_table.columns.tolist())

31-Mar-2026 00:53:41 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
31-Mar-2026 00:53:41 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
31-Mar-2026 00:53:41 INFO GEOparse - Parsing ./GPL570.txt: 
INFO:GEOparse:Parsing ./GPL570.txt: 
31-Mar-2026 00:53:41 DEBUG GEOparse - PLATFORM: GPL570
DEBUG:GEOparse:PLATFORM: GPL570


['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type', 'Sequence Source', 'Target Description', 'Representative Public ID', 'Gene Title', 'Gene Symbol', 'ENTREZ_GENE_ID', 'RefSeq Transcript ID', 'Gene Ontology Biological Process', 'Gene Ontology Cellular Component', 'Gene Ontology Molecular Function']


In [ ]:
# reset index
expr18781_reset = expr18781_f.reset_index()

# rename FIRST column to ID (safe way)
expr18781_reset = expr18781_reset.rename(
    columns={expr18781_reset.columns[0]: "ID"}
)

# merge with annotation
expr18781_annot = expr18781_reset.merge(
    gpl570_map,
    on="ID",
    how="inner"
)

# get sample columns
sample_cols = expr18781_f.columns.tolist()

# collapse probes → genes
expr18781_gene = (
    expr18781_annot[[symbol_col] + sample_cols]
    .groupby(symbol_col)
    .mean()
)

# clean gene names
expr18781_gene.index = expr18781_gene.index.astype(str).str.upper().str.strip()

print(expr18781_gene.shape)

(0, 37)


In [ ]:
import GEOparse

# load Illumina platform
gpl10558 = GEOparse.get_GEO("GPL10558", destdir=".")
gpl10558_table = gpl10558.table

print(gpl10558_table.columns.tolist())

31-Mar-2026 01:00:52 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
31-Mar-2026 01:00:52 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
31-Mar-2026 01:00:52 INFO GEOparse - Parsing ./GPL10558.txt: 
INFO:GEOparse:Parsing ./GPL10558.txt: 
31-Mar-2026 01:00:52 DEBUG GEOparse - PLATFORM: GPL10558
DEBUG:GEOparse:PLATFORM: GPL10558


['ID', 'Species', 'Source', 'Search_Key', 'Transcript', 'ILMN_Gene', 'Source_Reference_ID', 'RefSeq_ID', 'Unigene_ID', 'Entrez_Gene_ID', 'GI', 'Accession', 'Symbol', 'Protein_Product', 'Probe_Id', 'Array_Address_Id', 'Probe_Type', 'Probe_Start', 'SEQUENCE', 'Chromosome', 'Probe_Chr_Orientation', 'Probe_Coordinates', 'Cytoband', 'Definition', 'Ontology_Component', 'Ontology_Process', 'Ontology_Function', 'Synonyms', 'Obsolete_Probe_Id', 'GB_ACC']


In [ ]:
import GEOparse

gpl10558 = GEOparse.get_GEO("GPL10558", destdir=".")
gpl10558_table = gpl10558.table

print(gpl10558_table.columns.tolist())

31-Mar-2026 01:03:53 DEBUG utils - Directory . already exists. Skipping.
DEBUG:GEOparse:Directory . already exists. Skipping.
31-Mar-2026 01:03:53 INFO GEOparse - File already exist: using local version.
INFO:GEOparse:File already exist: using local version.
31-Mar-2026 01:03:53 INFO GEOparse - Parsing ./GPL10558.txt: 
INFO:GEOparse:Parsing ./GPL10558.txt: 
31-Mar-2026 01:03:53 DEBUG GEOparse - PLATFORM: GPL10558
DEBUG:GEOparse:PLATFORM: GPL10558


['ID', 'Species', 'Source', 'Search_Key', 'Transcript', 'ILMN_Gene', 'Source_Reference_ID', 'RefSeq_ID', 'Unigene_ID', 'Entrez_Gene_ID', 'GI', 'Accession', 'Symbol', 'Protein_Product', 'Probe_Id', 'Array_Address_Id', 'Probe_Type', 'Probe_Start', 'SEQUENCE', 'Chromosome', 'Probe_Chr_Orientation', 'Probe_Coordinates', 'Cytoband', 'Definition', 'Ontology_Component', 'Ontology_Process', 'Ontology_Function', 'Synonyms', 'Obsolete_Probe_Id', 'GB_ACC']


In [ ]:
possible_cols = ["Symbol", "Gene Symbol", "ILMN_Gene", "SYMBOL"]
symbol_col = None

for col in possible_cols:
    if col in gpl10558_table.columns:
        symbol_col = col
        break

print("Using symbol column:", symbol_col)

gpl10558_map = gpl10558_table[["ID", symbol_col]].copy()
gpl10558_map = gpl10558_map.dropna()
gpl10558_map = gpl10558_map[gpl10558_map[symbol_col].astype(str).str.strip() != ""]

gpl10558_map[symbol_col] = (
    gpl10558_map[symbol_col]
    .astype(str)
    .str.split("///")
    .str[0]
    .str.strip()
)

print(gpl10558_map.head())
print(gpl10558_map.shape)

Using symbol column: Symbol
             ID                   Symbol
0  ILMN_1343048      phage_lambda_genome
1  ILMN_1343049      phage_lambda_genome
2  ILMN_1343050  phage_lambda_genome:low
3  ILMN_1343052  phage_lambda_genome:low
4  ILMN_1343059                     thrB
(44837, 2)


In [ ]:
expr42834_reset = expr42834_f.reset_index()
expr42834_reset = expr42834_reset.rename(columns={expr42834_reset.columns[0]: "ID"})

expr42834_annot = expr42834_reset.merge(
    gpl10558_map,
    on="ID",
    how="inner"
)

sample_cols_42834 = expr42834_f.columns.tolist()

expr42834_gene = (
    expr42834_annot[[symbol_col] + sample_cols_42834]
    .groupby(symbol_col)
    .mean()
)

expr42834_gene.index = expr42834_gene.index.astype(str).str.upper().str.strip()

print(expr42834_gene.shape)

(31426, 251)


In [ ]:
expr83456_reset = expr83456_f.reset_index()
expr83456_reset = expr83456_reset.rename(columns={expr83456_reset.columns[0]: "ID"})

expr83456_annot = expr83456_reset.merge(
    gpl10558_map,
    on="ID",
    how="inner"
)

sample_cols_83456 = expr83456_f.columns.tolist()

expr83456_gene = (
    expr83456_annot[[symbol_col] + sample_cols_83456]
    .groupby(symbol_col)
    .mean()
)

expr83456_gene.index = expr83456_gene.index.astype(str).str.upper().str.strip()

print(expr83456_gene.shape)

(31334, 110)


In [ ]:
shared_genes = sorted(
    set(expr18781_gene.index) &
    set(expr42834_gene.index) &
    set(expr83456_gene.index)
)

print("Shared genes:", len(shared_genes))

Shared genes: 0


In [ ]:
print("18781 gene examples:", list(expr18781_gene.index[:20]))
print("42834 gene examples:", list(expr42834_gene.index[:20]))
print("83456 gene examples:", list(expr83456_gene.index[:20]))

print("\n18781 shape:", expr18781_gene.shape)
print("42834 shape:", expr42834_gene.shape)
print("83456 shape:", expr83456_gene.shape)

print("\n18781 vs 42834 overlap:", len(set(expr18781_gene.index) & set(expr42834_gene.index)))
print("18781 vs 83456 overlap:", len(set(expr18781_gene.index) & set(expr83456_gene.index)))
print("42834 vs 83456 overlap:", len(set(expr42834_gene.index) & set(expr83456_gene.index)))

18781 gene examples: []
42834 gene examples: ['1-DEC', '1-MAR', '10-MAR', '11-MAR', '2-MAR', '3-MAR', '4-MAR', '5-MAR', '6-MAR', '7-MAR', '7A5', '8-MAR', '9-MAR', 'A1BG', 'A1CF', 'A26C3', 'A2BP1', 'A2LD1', 'A2M', 'A2ML1']
83456 gene examples: ['1-DEC', '1-MAR', '10-MAR', '11-MAR', '2-MAR', '3-MAR', '4-MAR', '5-MAR', '6-MAR', '7-MAR', '7A5', '8-MAR', '9-MAR', 'A1BG', 'A1CF', 'A26C3', 'A2BP1', 'A2LD1', 'A2M', 'A2ML1']

18781 shape: (0, 37)
42834 shape: (31426, 251)
83456 shape: (31334, 110)

18781 vs 42834 overlap: 0
18781 vs 83456 overlap: 0
42834 vs 83456 overlap: 31332


In [ ]:
# rebuild GPL570 map
symbol_col_18781 = "Gene Symbol"

gpl570_map = gpl570_table[["ID", symbol_col_18781]].copy()
gpl570_map = gpl570_map.dropna()
gpl570_map = gpl570_map[gpl570_map[symbol_col_18781].astype(str).str.strip() != ""]

gpl570_map[symbol_col_18781] = (
    gpl570_map[symbol_col_18781]
    .astype(str)
    .str.split("///").str[0]
    .str.split("//").str[0]
    .str.split(";").str[0]
    .str.strip()
    .str.upper()
)

# reset expression and force probe IDs to lowercase to match GPL570
expr18781_reset = expr18781_f.reset_index()
expr18781_reset = expr18781_reset.rename(columns={expr18781_reset.columns[0]: "ID"})
expr18781_reset["ID"] = expr18781_reset["ID"].astype(str).str.lower().str.strip()

gpl570_map["ID"] = gpl570_map["ID"].astype(str).str.lower().str.strip()

expr18781_annot = expr18781_reset.merge(
    gpl570_map,
    on="ID",
    how="inner"
)

sample_cols_18781 = expr18781_f.columns.tolist()

expr18781_gene = (
    expr18781_annot[[symbol_col_18781] + sample_cols_18781]
    .groupby(symbol_col_18781)
    .mean()
)

expr18781_gene.index = expr18781_gene.index.astype(str).str.strip().str.upper()

print(expr18781_gene.shape)
print(list(expr18781_gene.index[:20]))

(22878, 37)
['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A2MP1', 'A4GALT', 'A4GNT', 'AA06', 'AAAS', 'AACS', 'AACSP1', 'AADAC', 'AADACL2', 'AADACP1', 'AADAT', 'AAED1', 'AAGAB', 'AAK1']


In [ ]:
print("18781 shape:", expr18781_gene.shape)
print("42834 shape:", expr42834_gene.shape)
print("83456 shape:", expr83456_gene.shape)

print("18781 vs 42834 overlap:", len(set(expr18781_gene.index) & set(expr42834_gene.index)))
print("18781 vs 83456 overlap:", len(set(expr18781_gene.index) & set(expr83456_gene.index)))
print("42834 vs 83456 overlap:", len(set(expr42834_gene.index) & set(expr83456_gene.index)))

18781 shape: (22878, 37)
42834 shape: (31426, 251)
83456 shape: (31334, 110)
18781 vs 42834 overlap: 16622
18781 vs 83456 overlap: 16622
42834 vs 83456 overlap: 31332


In [ ]:
shared_genes = sorted(
    set(expr18781_gene.index) &
    set(expr42834_gene.index) &
    set(expr83456_gene.index)
)

print("Shared genes:", len(shared_genes))

Shared genes: 16622


In [ ]:
# collapse any duplicate gene names
expr18781_gene = expr18781_gene.groupby(expr18781_gene.index).mean()
expr42834_gene = expr42834_gene.groupby(expr42834_gene.index).mean()
expr83456_gene = expr83456_gene.groupby(expr83456_gene.index).mean()

print("18781 unique:", expr18781_gene.index.is_unique)
print("42834 unique:", expr42834_gene.index.is_unique)
print("83456 unique:", expr83456_gene.index.is_unique)

18781 unique: True
42834 unique: True
83456 unique: True


In [ ]:
shared_genes = sorted(
    set(expr18781_gene.index) &
    set(expr42834_gene.index) &
    set(expr83456_gene.index)
)

print("Shared genes:", len(shared_genes))

Shared genes: 16622


In [ ]:
X_all = pd.concat([
    expr18781_gene.loc[shared_genes],
    expr42834_gene.loc[shared_genes],
    expr83456_gene.loc[shared_genes]
], axis=1)

y_all = pd.concat([y18781, y42834, y83456])

X_all = X_all.T
X_all = X_all.loc[y_all.index]

print(X_all.shape)
print(y_all.value_counts())

(398, 16622)
0    229
1    169
Name: count, dtype: int64


In [ ]:
X_all.to_csv("sarcoidosis_merged_expression_398x16622.csv")
y_all.to_csv("sarcoidosis_merged_labels.csv", header=True)

sample_manifest = pd.DataFrame({
    "Sample": X_all.index,
    "Label": y_all.values
})
sample_manifest.to_csv("sarcoidosis_sample_manifest.csv", index=False)

In [ ]:
X_all.to_csv("sarcoidosis_expression_final.csv")
y_all.to_csv("sarcoidosis_labels_final.csv", header=True)

In [ ]:
X_all.to_csv("sarcoidosis_expression_final.csv")
y_all.to_csv("sarcoidosis_labels_final.csv", header=True)

In [ ]:
from google.colab import files
files.download("sarcoidosis_expression_final.csv")
files.download("sarcoidosis_labels_final.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>